In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

In [2]:
routes = pd.read_csv("../data/raw/route.csv")

In [3]:
G = nx.DiGraph()

In [4]:
# bus stops
stop_map = {
    # "หอพักเอเชียนเกม (B Zone)": "Dorm",
    "หอพักเอเชียนเกม": "Dorm",
    "green": "Green",
    "SC2 SC3": "SC2_SC3",
    "อาคารสุขศาสตร์": "Health",
    # "อาคารบรรยายรวม 4": "Lecture",
    "อาคารบรรยายรวม": "Lecture",
    "ศูนย์ประชุม": "Convention",
    "สถานีขนส่ง": "Terminal",
    "หอสมุดป๋วย": "Library",
    "รพ.ธรรมศาสตร์": "Hospital",
    "อุทยานการเรียนรู้ป๋วย 100 ปี": "Park",
    # "อาคารบรรยายรวม 3": "Lecture3",
    # "อาคารบรรยายรวม 1": "Lecture1",
    "TU DOME": "Dome",
    "ประตูเชียงราก 1": "Gate1",
    "SC1": "SC1",
    "อาคารเรียนรวมกลุ่มสังคมศาสตร์": "Social"
}

stops = list(set(stop_map.values()))
G.add_nodes_from(stops)


### Convert KM to Minute

In [5]:
def km_to_min(km):
    return round(km * 3)  # 20 km/h

# เสริมพิเศษ
G.add_edge("Dorm", "Green", time=km_to_min(1.2))
G.add_edge("Green", "SC2_SC3", time=km_to_min(0.6))
G.add_edge("SC2_SC3", "Health", time=km_to_min(1.0))
G.add_edge("Health", "Lecture", time=km_to_min(0.5))
G.add_edge("Lecture", "Dorm", time=km_to_min(1.9))

# สีแดง 1A
G.add_edge("Convention", "Terminal", time=km_to_min(1.1))
G.add_edge("Terminal", "SC2_SC3", time=km_to_min(0.65))
G.add_edge("SC2_SC3", "Library", time=km_to_min(0.6))
G.add_edge("Library", "Dorm", time=km_to_min(1.5))
G.add_edge("Dorm", "Convention", time=km_to_min(3.6))

# สีเหลือง 1B
G.add_edge("Convention", "Dorm", time=km_to_min(3.6))
G.add_edge("Dorm", "Library", time=km_to_min(1.5))
G.add_edge("Library", "SC2_SC3", time=km_to_min(0.6))
G.add_edge("SC2_SC3", "Terminal", time=km_to_min(0.65))
G.add_edge("Terminal", "Convention", time=km_to_min(1.1))

# สีเขียว 2
G.add_edge("Convention", "Hospital", time=km_to_min(0.7))
G.add_edge("Hospital", "Health", time=km_to_min(0.5))
G.add_edge("Health", "Park", time=km_to_min(0.1))
G.add_edge("Park", "Lecture", time=km_to_min(0.28))
# G.add_edge("Lecture", "Lecture3", time=km_to_min(0.22))
# G.add_edge("Lecture3", "Lecture1", time=km_to_min(0.45))
G.add_edge("Lecture1", "Convention", time=3)  # สมมติ

# สีม่วง 3
G.add_edge("Dome", "Gate1", time=3)
G.add_edge("Gate1", "SC1", time=3)
G.add_edge("SC1", "Library", time=3)
G.add_edge("Library", "Green", time=3)
G.add_edge("Green", "Dorm", time=3)

# สีฟ้า 5
G.add_edge("Convention", "Social", time=3)
G.add_edge("Social", "Lecture", time=3)
G.add_edge("Lecture", "Park", time=3)
G.add_edge("Park", "Health", time=3)
G.add_edge("Health", "Hospital", time=3)

### Bidirectional

In [6]:
edges = list(G.edges(data=True))

for u, v, data in edges:
    G.add_edge(v, u, time=data["time"])

### Route for simulation

In [7]:
route_special = ["Dorm", "Green", "SC2_SC3", "Health", "Lecture", "Dorm"]
route_red = ["Convention", "Terminal", "SC2_SC3", "Library", "Dorm", "Convention"]
route_yellow = ["Convention", "Dorm", "Library", "SC2_SC3", "Terminal", "Convention"]
route_purple = ["Dome", "Gate1", "SC1", "Library", "Green", "Dorm"]
route_blue = ["Convention", "Social", "Lecture", "Park", "Health", "Hospital"]
route_green = ["Convention", "Hospital", "Health", "Park", "Lecture", "Convention"]

### Dijkstra Algorithm

In [8]:
path = nx.shortest_path(G, "SC2_SC3", "Hospital", weight="time")
time = nx.shortest_path_length(G, "SC2_SC3", "Hospital", weight="time")
print(path , time)

['SC2_SC3', 'Health', 'Hospital'] 5


### Bus Object Class

In [9]:
class Bus:
    def __init__(self, id, capacity, route):
        self.id = id
        self.capacity = capacity
        self.route = route
        self.current_index = 0   # อยู่ป้ายไหน
        self.passengers = []

### Define Buses

In [10]:
buses = [
    Bus(0, 30, route_special),
    Bus(1, 30, route_red),
    Bus(2, 30, route_yellow),
    Bus(3, 30, route_purple),
    Bus(4, 30, route_blue),
    Bus(5, 30, route_green),
]

### Passenger Model

In [11]:
class Passenger:
    def __init__(self, origin, destination, arrival_time):
        self.origin = origin
        self.destination = destination
        self.arrival_time = arrival_time

### Queue

In [12]:
queues = {
    "Dorm": [],
    "Green": [],
    "SC2_SC3": [],
    "Health": [],
    "Lecture": [],
    "Terminal" : [],
    "Convention" : [],
    "Library" : [],
    "SC1" : [],
    "Gate1" : [],
    "Hospital" : [],
    "Social" : [],
    "Park" : [],
    "Dome" : []
}

In [13]:
waiting_times = []
queue_lengths = []

### Passenger Demand

In [14]:
import random

def generate_passengers(current_time):
    if random.random() < 0.5:  # เพิ่ม demand

        origin = random.choice(stops)
        destination = random.choice(stops)

        # ❗ ห้าม origin == destination
        while destination == origin:
            destination = random.choice(stops)

        p = Passenger(origin, destination, current_time)
        queues[origin].append(p)

### Move Bus

In [15]:
def move_bus(bus):
    bus.current_index += 1
    if bus.current_index >= len(bus.route):
        bus.current_index = 0

### Passenger Boarding

In [16]:
def board_passengers(bus,t):
    current_stop = bus.route[bus.current_index]
    queue = queues[current_stop]

    while queue and len(bus.passengers) < bus.capacity:
        p = queue.pop(0)
        bus.passengers.append(p)

        print("Passenger boarded:", p.origin, "→", p.destination)
        waiting_time = t - p.arrival_time
        waiting_times.append(waiting_time)

### Passenger Down

In [17]:
def drop_passengers(bus):
    current_stop = bus.route[bus.current_index]
    bus.passengers = [
        p for p in bus.passengers if p.destination != current_stop
    ]

### Simulation Loop

In [ ]:
simulation_time = 60

for t in range(simulation_time):
    
    generate_passengers(t)

    for bus in buses:
        move_bus(bus)
        drop_passengers(bus)
        board_passengers(bus,t)

Passenger boarded: Green → Park
Passenger boarded: Library → Green
Passenger boarded: Social → Library
Passenger boarded: SC2_SC3 → Library
Passenger boarded: Health → Park
Passenger boarded: Dome → Dorm
Passenger boarded: Hospital → Gate1
Passenger boarded: Park → Dorm
Passenger boarded: Gate1 → Green
Passenger boarded: SC2_SC3 → SC1
Passenger boarded: SC2_SC3 → Green
Passenger boarded: SC1 → Lecture
Passenger boarded: Dome → Park
Passenger boarded: Lecture → Library
Passenger boarded: Health → SC1
Passenger boarded: Dorm → Health
Passenger boarded: Terminal → Green
Passenger boarded: Social → Library
Passenger boarded: SC1 → Health
Passenger boarded: Dorm → SC1
Passenger boarded: Dome → Library
Passenger boarded: SC1 → Dorm
Passenger boarded: Health → Dorm
Passenger boarded: Health → Dome
Passenger boarded: Lecture → SC1
Passenger boarded: Dorm → Green
Passenger boarded: Green → Library
Passenger boarded: Library → Hospital
Passenger boarded: Dome → Library
Passenger boarded: Gate1 →

### Queue Collection

In [19]:
total_queue = sum(len(q) for q in queues.values())
queue_lengths.append(total_queue)

### Result 

In [ ]:
avg_wait = sum(waiting_times) / len(waiting_times)
print(f"Average waiting time: {avg_wait} minute" ) # minutes

Average waiting time: 1.8857142857142857 minute
minute
